# Orbital Debris SQL Queries

This notebook is intentionally SQL-first and contains query work only (no visualizations).

## **Setup And Function Declarations**

In [1]:
import pandas as pd
import sqlite3 as sql
import utility as utils

orbital_debris_conn = sql.connect('../data/clean/orbital_debris.db')

# Sanity Check - Verify that the database connection is working and that we can retrieve data
sanity_check = utils.query_all_satellites(orbital_debris_conn)

display(sanity_check)

,norad_id,cospar_id,object_name,launch_year,launch_date,launch_site,owner,country_operator,object_type
0,5,1958-002B,VANGUARD 1,1958,1958-03-17,AFETR,US,USA,PAYLOAD
1,11,1959-001A,VANGUARD 2,1959,1959-02-17,AFETR,US,USA,PAYLOAD
2,12,1959-001B,VANGUARD R/B,1959,1959-02-17,AFETR,US,USA,ROCKET BODY
3,16,1958-002A,VANGUARD R/B,1958,1958-03-17,AFETR,US,USA,ROCKET BODY
4,20,1959-007A,VANGUARD 3,1959,1959-09-18,AFETR,US,USA,PAYLOAD
...,...,...,...,...,...,...,...,...,...
33229,68201,2026-052D,OBJECT D,2026,2026-03-16,JSC,PRC,CHINA,PAYLOAD
33230,68202,2026-052E,OBJECT E,2026,2026-03-16,JSC,PRC,CHINA,PAYLOAD
33231,68203,2026-052F,OBJECT F,2026,2026-03-16,JSC,PRC,CHINA,PAYLOAD
33232,68204,2026-052G,OBJECT G,2026,2026-03-16,JSC,PRC,CHINA,PAYLOAD


## Primary Question 1: Growth and Decoupling Baseline
At what year does the present day in-orbit population stop following a legacy linear pattern and transition into a modern accelerating pattern?

In [2]:
# CTE yearly aggregates the number of objects in orbit, launch missions, and payloads in orbit by launch year,
# then calculates the share of the payloads and the cumulative number of objects in orbit over time.
q1 = '''
WITH yearly AS (
  SELECT
    launch_events.launch_year AS launch_year,
    COUNT(*) AS objects_in_orbit,
    COUNT(DISTINCT launch_events.launch_id) AS launch_missions,
    SUM(
      CASE
        WHEN UPPER(COALESCE(satellites.object_type, '')) = 'PAYLOAD' THEN 1
        ELSE 0
      END
    ) AS payload_in_orbit
  FROM satellites
  JOIN launch_events ON launch_events.launch_id = satellites.launch_id
  WHERE launch_events.launch_year IS NOT NULL
    AND COALESCE(satellites.in_orbit, 0) = 1
  GROUP BY launch_events.launch_year
)
SELECT
  launch_year,
  objects_in_orbit,
  launch_missions,
  payload_in_orbit,
  ROUND(100.0 * payload_in_orbit / NULLIF(objects_in_orbit, 0), 2) AS payload_share_pct,
  SUM(objects_in_orbit) OVER (ORDER BY launch_year) AS cumulative_in_orbit
FROM yearly
ORDER BY launch_year;
'''
pq1 = utils.run_query(q1, orbital_debris_conn)
pq1.to_parquet('../data/clean/results/pq1_orbit_trends.parquet', index=False)
pq1.head(10)

,launch_year,objects_in_orbit,launch_missions,payload_in_orbit,payload_share_pct,cumulative_in_orbit
0,1958,3,1,1,33.33,3
1,1959,7,5,5,71.43,10
2,1960,13,6,5,38.46,23
3,1961,212,8,9,4.25,235
4,1962,33,15,14,42.42,268
5,1963,91,12,17,18.68,359
6,1964,60,21,28,46.67,419
7,1965,449,33,52,11.58,868
8,1966,207,34,38,18.36,1075
9,1967,95,31,49,51.58,1170


## **Primary Question 2: High-Risk Distribution by Altitude Band**
How are high-risk objects (by velocity and kinetic energy) distributed across orbit classes and altitude bands, especially in the 400 - 600 km LEO?

In [3]:
# This query is a bit more complex, but it is designed to identify the highest risk objects in orbit 
# based on their kinetic energy.
#
# CTE banded: joins the orbital_data, risk_assessment, and satellites tables to
# get all of the relevent information/fields in one place and we create a new field called altitude_band
# that categorizes each object into one of the 5 altitude bands based on their perigee_km.
#
# CTE ranked: uses the NTILE window function to divide the objects into 4 quartiles based on their kinetic energy.
#   QUARTILE 1 = highest kinetic energy (most dangerous, top 25% of objects),
#   QUARTILE 2 = next highest kinetic energy (second most dangerous, 25-50% of objects),
#   QUARTILE 3 = moderate kinetic energy (50-75% of objects), 
#   QUARTILE 4 = lowest kinetic energy (least dangerous, bottom 25% of objects)
# Finally we select from ranked and group by orbit_class and altitude_band 
# to get the total number of objects, the number of high risk objects (those in the top quartile), the percentage 
# of high risk objects, the average velocity and kinetic energy of the high risk objects, the total kinetic energy
# of the high risk objects, and the number of high risk objects that are zombies, debris, payloads, or rocket bodies.

q2 = '''
WITH banded AS (
  SELECT
    orbital_data.norad_id,
    orbital_data.orbit_class,
    orbital_data.perigee_km,
    orbital_data.proxy_mass_kg,
    risk_assessment.velocity_kms,
    risk_assessment.kinetic_joules,
    risk_assessment.is_zombie,
    satellites.object_type,
    CASE
      WHEN orbital_data.perigee_km < 400 THEN '1. < 400 km'
      WHEN orbital_data.perigee_km >= 400  AND orbital_data.perigee_km < 600 THEN '2. 400-600 km (Kessler Zone)'
      WHEN orbital_data.perigee_km >= 600  AND orbital_data.perigee_km < 1000 THEN '3. 600-1000 km'
      WHEN orbital_data.perigee_km >= 1000 AND orbital_data.perigee_km < 2000 THEN '4. 1000-2000 km'
      ELSE '5. > 2000 km'
    END AS altitude_band
  FROM orbital_data
  JOIN risk_assessment ON risk_assessment.norad_id = orbital_data.norad_id
  JOIN satellites ON satellites.norad_id = orbital_data.norad_id
  WHERE risk_assessment.kinetic_joules IS NOT NULL
    AND orbital_data.perigee_km IS NOT NULL
),
ranked AS (
  SELECT
    *,
    NTILE(4) OVER (ORDER BY kinetic_joules DESC) AS kinetic_quartile
  FROM banded
)
SELECT
  orbit_class,
  altitude_band,

  COUNT(*) AS total_objects,
  SUM(CASE WHEN kinetic_quartile = 1 THEN 1 ELSE 0 END) AS high_risk_count,
  
  ROUND(100.0 * SUM(CASE WHEN kinetic_quartile = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS high_risk_pct,
  ROUND(AVG(CASE WHEN kinetic_quartile = 1 THEN velocity_kms  END), 4) AS avg_velocity_kms,
  ROUND(AVG(CASE WHEN kinetic_quartile = 1 THEN kinetic_joules END), 0) AS avg_kinetic_joules,
  ROUND(SUM(CASE WHEN kinetic_quartile = 1 THEN kinetic_joules ELSE 0 END), 0) AS total_kinetic_joules,
  
  SUM(CASE WHEN kinetic_quartile = 1 AND is_zombie = 1 THEN 1 ELSE 0 END) AS zombie_high_risk,
  SUM(CASE WHEN kinetic_quartile = 1 AND UPPER(object_type) = 'DEBRIS'  THEN 1 ELSE 0 END) AS debris_high_risk,
  SUM(CASE WHEN kinetic_quartile = 1 AND UPPER(object_type) = 'PAYLOAD' THEN 1 ELSE 0 END) AS payload_high_risk,
  SUM(CASE WHEN kinetic_quartile = 1 AND UPPER(object_type) = 'ROCKET BODY' THEN 1 ELSE 0 END) AS rb_high_risk
FROM ranked
GROUP BY orbit_class, altitude_band
ORDER BY altitude_band, orbit_class;
'''

high_risk = utils.run_query(q2, orbital_debris_conn)
high_risk.to_parquet('../data/clean/results/pq2_high_risk.parquet', index=False)
high_risk.head(10)

,orbit_class,altitude_band,total_objects,high_risk_count,high_risk_pct,avg_velocity_kms,avg_kinetic_joules,total_kinetic_joules,zombie_high_risk,debris_high_risk,payload_high_risk,rb_high_risk
0,ELLIPTICAL,1. < 400 km,3,0,0.00,NaN,NaN,0.000000e+00,0,0,0,0
1,LEO,1. < 400 km,1704,1304,76.53,7.6915,1.413056e+10,1.842625e+13,63,0,1252,52
2,MEO,1. < 400 km,643,339,52.72,4.6688,2.224586e+10,7.541345e+12,1,0,1,338
3,GEO,2. 400-600 km (Kessler Zone),1,0,0.00,NaN,NaN,0.000000e+00,0,0,0,0
4,LEO,2. 400-600 km (Kessler Zone),12328,4126,33.47,7.6212,1.750860e+10,7.224047e+13,137,0,3894,232
5,MEO,2. 400-600 km (Kessler Zone),591,76,12.86,4.4262,2.015823e+10,1.532025e+12,0,0,0,76
6,UNKNOWN,2. 400-600 km (Kessler Zone),597,410,68.68,7.6262,3.167457e+10,1.298657e+13,149,0,227,183
7,ELLIPTICAL,3. 600-1000 km,4,0,0.00,NaN,NaN,0.000000e+00,0,0,0,0
8,LEO,3. 600-1000 km,9552,774,8.10,7.4305,5.269667e+10,4.078722e+13,117,6,229,539
9,MEO,3. 600-1000 km,553,80,14.47,4.4471,2.049217e+10,1.639373e+12,2,0,3,77


## **Primary Question 3: Zombie Concentration by Owner**

## **Secondary: Object Type × Operational Status**

## **Secondary: User Category Profile**

## **Extra Questions!**